<a href="https://colab.research.google.com/github/soledad-villarrubia/Data_Sciense_3/blob/main/Proyecto_Final_Ciencia_de_Datos_IIi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Final Data Science III

# Análisis de sentimiento en reseñas de películas en español utilizando técnicas de NLP y Machine Learning

## 1. Introducción

En este trabajo se desarrolla un análisis de sentimiento sobre reseñas de películas escritas en español, utilizando técnicas de procesamiento de lenguaje natural (NLP) y modelos de machine learning.

El objetivo principal es identificar automáticamente si una reseña expresa un sentimiento positivo o negativo. Para ello, se trabaja con un dataset de Filmaffinity obtenido desde Kaggle, que contiene opiniones reales de usuarios y un puntaje asociado a cada reseña.

A lo largo del trabajo se realizará la carga y exploración de los datos, la limpieza del texto, la construcción de una variable objetivo, la vectorización del lenguaje y el entrenamiento de un modelo de clasificación. Finalmente, se evaluarán los resultados obtenidos y se presentarán conclusiones y posibles mejoras futuras.

In [32]:
import requests

url = "https://raw.githubusercontent.com/soledad-villarrubia/Data_Sciense_3/refs/heads/main/reviews_filmaffinity.csv"


response = requests.get(url)  # Hace una solicitud GET para obtener el contenido del archivo
print(response.text[:300])  # Imprime los primeros 300 caracteres del archivo para ver cómo está delimitado

df = pd.read_csv(url, sep=';', encoding='latin1')

df = df.loc[:, ~df.columns.str.contains('^Unnamed')].copy()

# The original data does not contain a column named 'tweet_id,label,tweet_text'.
# Instead, it contains 'review_text' and 'review_rate'.
# We will rename 'review_text' to 'tweet_text' and 'review_rate' to 'label'
# to align with the subsequent processing steps.
df.rename(columns={'review_text': 'tweet_text', 'review_rate': 'label'}, inplace=True)



df['label'] = pd.to_numeric(df['label'], errors='coerce')

df.dropna(inplace=True)

df.head()

film_name;gender;film_avg_rate;review_rate;review_title;review_text;;;;;
Ocho apellidos vascos;Comedia;6;3;OCHO APELLIDOS VASCOS...Y NINGÚN NOMBRE PROPIO;"La mayor virtud de esta película es su existencia.El hecho de que podamos jugar con los tópicos más extremos de las identidades patrias (la anda


,film_name,gender,film_avg_rate,label,review_title,tweet_text
0,Ocho apellidos vascos,Comedia,6,3.0,OCHO APELLIDOS VASCOS...Y NINGÃN NOMBRE PROPIO,La mayor virtud de esta pelÃ­cula es su existe...
1,Ocho apellidos vascos,Comedia,6,2.0,El perro verde,"No soy un experto cinÃ©filo, pero pocas veces ..."
2,Ocho apellidos vascos,Comedia,6,2.0,Si no eres de comer mierda... no te comas esta...,Si no eres un incondicional del humor estilo T...
3,Ocho apellidos vascos,Comedia,6,2.0,Aida: The movie,"No sÃ© quÃ© estÃ¡ pasando, si la gente se deja..."
4,Ocho apellidos vascos,Comedia,6,2.0,UN HOMBRE SOLO (Julio Iglesias 1987),"Pero cuando amanece,y me quedo solo,siento en ..."


In [33]:
# IMPORTACIÓN DE LIBRERÍAS
# ==========================================
import os
import re
import random
import warnings
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# CONFIGURACIÓN GENERAL

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Recursos de NLTK
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
